# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and is fully compatible with the FAIR principles.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}\n")
print("Cite as:", getattr(metadata, 'cite_as', None) or getattr(metadata, 'citeAs', None) or metadata.__dict__.get('citeAs', None))
print(f"License: {metadata.license}\n")
print("Published:", getattr(metadata, 'date_published', None) or getattr(metadata, 'datePublished', None) or metadata.__dict__.get('datePublished', None))

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We use `mlcroissant` programmatic APIs to inspect available record sets (tables) and their fields. Each entity is referenced by its `@id`.

In [ ]:
# List available record sets and their fields
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record sets:\n")

for rs in record_sets:
    print(f"Record Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print("  Fields:")
    for fld in rs.fields:
        print(f"    Field Name: {fld.name.ljust(24)} @id: {fld.id} @type: {fld.data_type if hasattr(fld, 'data_type') else ''}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the record set and field `@id`s from the overview.

We will extract the main protein abundance table (record set) from the dataset. Be sure to use the proper RecordSet `@id`. (If you are running this notebook, copy the RecordSet `@id` you want to explore below.)

In [ ]:
# Extract all data from each record set as pandas DataFrames

dataframes = {}

for rs in dataset.record_sets:
    print(f"Loading record set: {rs.name} (@id: {rs.id}) ...")
    try:
        records = list(dataset.records(record_set=rs.id))
        df = pd.DataFrame(records)
        dataframes[rs.id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(f"  Sample data:")
        display(df.head(2))
    except Exception as e:
        print(f"  Failed to load records: {e}")
    print()
# Select a primary record set for further analysis. Update this value if needed.
main_record_set_id = list(dataframes.keys())[0] if dataframes else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on a specific numeric field, normalizing values, and grouping the data.
All field (column) references are by `@id`. For human-readable analysis, refer to the previous overview for mapping to display names.

In [ ]:
# --- Specify the field `@id`s for one numeric column and one categorical column, as discovered above ---
# Replace these IDs with those from the overview if you wish to analyze other fields.

df = dataframes[main_record_set_id]

# If you are unsure, print columns:
print("Available columns:", df.columns.tolist())

# Example (update to match real field @id):
numeric_field_id = [col for col in df.columns if 'abundance' in col.lower() or 'coverage' in col.lower() or col.lower() in ('mw', 'molecular_weight')]
if not numeric_field_id:
    numeric_field_id = [df.columns[0]]  # Default to first if heuristic fails
numeric_field_id = numeric_field_id[0]

group_field_id = [col for col in df.columns if 'description' in col.lower() or 'category' in col.lower() or 'accession' in col.lower()]
if not group_field_id:
    group_field_id = [df.columns[1]]  # Default to second if heuristic fails
group_field_id = group_field_id[0]

print(f'Will filter and analyze on numeric field: {numeric_field_id} and group by: {group_field_id}')

# Ensure the field is numeric
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filter: select records where numeric_field > threshold
threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id].mean()) else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} records\n")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[numeric_field_id + '_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized field ({numeric_field_id}):")
display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

# Group by the group_field and compute average
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields.

This section plots distributions of the selected numeric field and a simple group-wise aggregation.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(6, 4))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If grouped_df exists and is not empty, bar plot the top N groupings
if 'grouped_df' in locals() and not grouped_df.empty:
    plt.figure(figsize=(10, 4))
    top_group = grouped_df.sort_values(numeric_field_id, ascending=False).head(20)
    sns.barplot(x=group_field_id, y=numeric_field_id, data=top_group)
    plt.xticks(rotation=90)
    plt.title(f'Mean {numeric_field_id} by {group_field_id} (top 20)')
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, and analyze the [FAIR^2 Mass Spectrometry dataset](https://doi.org/10.71728/senscience.vq4a-28xa) using `mlcroissant`, referencing all core data entities by their `@id` fields for reproducibility.

You can reuse this notebook template for any Croissant-compatible dataset by modifying record set and field `@id`s according to the dataset's schema.